# BUSI2066 · Week 5 — Describing Data


### Before you start

1. Open your Codespace, and drag in **this notebook**, **`estore.csv`** and
   **`loans_income.csv`**. All three must sit next to each other.
2. **Select the kernel** — top right.

### Today

Week 4 gave you one chart for one column. Today you get the **numbers** behind those charts —
the centre, the spread, and the shape — and you learn to compare **one column across groups**,
which is what almost every research question in your project actually asks for.

### The data

`estore.csv` — **1,000 visits to an online store**, one row per customer.

| Column | What it records |
|---|---|
| `country`, `city` | where the customer was |
| `revisit` | 0 = new visitor · 1 = came back |
| `storeNum` | the last store they visited — an **ID**, not a quantity |
| `pageViews` | pages viewed on the visit |
| `items` | items bought on the visit |
| `revenue` | money spent on the visit |
| `satisfaction` | how satisfied they were, on a five-point scale |

### Today, in two sections

| | Teaching, then Python | Your questions |
|---|---|---|
| **Section 1 · One column** | Parts 1–4 — counts, order, centre, spread, charts | **1–4** |
| ☕ **Break** | | |
| **Section 2 · Groups and shape** | Parts 5–6 — groups, skewness, the QQ plot | **5–10** |

### The rule

> **AI explains. It does not write** — except where marked ⭐.

Two exceptions today, both marked. Chart styling is one of them: there are a dozen options and
memorising them is a waste of your time.

---
# Part 0 · Set up

Same three lines as always, plus the two you will need later.

In [ ]:
%pip install -q pandas seaborn scipy statsmodels

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

df = pd.read_csv('estore.csv')

print(df.shape)
df.head()

Four imports now. Two are new:

| Import | What it is for |
|---|---|
| `matplotlib.pyplot as plt` | The layer underneath seaborn. You need it to **add titles and labels** |
| `scipy.stats as stats` | Statistics. Today: skewness, kurtosis and the QQ plot |

`seaborn` draws the chart; `matplotlib` lets you label it. Every chart you make from now on uses both.

---
---
# 📘 SECTION 1 · One column

**Parts 1 to 4, then questions 1 to 4.**

Everything in this section describes **one column at a time**.

---
# Part 1 · Counting, and turning counts into probability

You met `value_counts()` in week 3. It has one option that matters.

In [ ]:
df['country'].value_counts()

Before counting, it is often worth just asking **which values exist at all**.

In [ ]:
df['country'].unique()

`unique()` lists each distinct value once, with no counts. On a column you have never seen it is the
fastest way to find out what you are dealing with — and to spot two spellings of the same answer,
which week 4 warned you about.

Use `unique()` to see **what is there**, and `value_counts()` to see **how much of each**.

Now the same thing as **shares** rather than counts.

In [ ]:
df['country'].value_counts(normalize=True)

`normalize=True` divides every count by the total, so the numbers add up to 1.

**That is a probability.** If you pick one visit from this file at random, there is a 0.325 chance
it came from the United States. Counts describe the file you have; probabilities let you talk about
the next customer you have not seen — which is the whole business of the second half of this course.

Percentages read better in a report. Multiply by 100 and round.

In [ ]:
(df['country'].value_counts(normalize=True) * 100).round(1)

---
# Part 2 · Telling pandas that a column has an order

`satisfaction` is **ordinal** — *very unsatisfied* is below *unsatisfied*, which is below
*neutral*, and so on. You know that. Pandas does not. To pandas it is just text.

In [ ]:
sns.countplot(data=df, y='satisfaction')
plt.show()

The bars come out in the order each answer **first appears in the file** — *very unsatisfied*,
then *neutral*, then whatever came next. For a nominal column like `country` that does no harm.
For an **ordinal** column it throws away the one thing that made it ordinal.

Fix it once, and every chart and table afterwards obeys the order.

In [ ]:
satisfaction_order = ['very unsatisfied', 'unsatisfied', 'neutral',
                      'satisfied', 'very satisfied']

df['satisfaction'] = pd.Categorical(df['satisfaction'],
                                    categories=satisfaction_order,
                                    ordered=True)

df['satisfaction'].value_counts(sort=False)

Two things to notice.

**`sort=False` is doing work here.** `value_counts()` sorts by size by default, and it keeps doing
that even after you have declared an order — so on an ordinal column you almost always want
`sort=False`, which says *leave them in category order*.

**Charts and `groupby` need no such flag.** They pick up the order on their own.

In [ ]:
sns.countplot(data=df, y='satisfaction')
plt.show()

⚠️ **This is the single most useful idea in today's notebook for your project.**

Every Likert item in your survey arrives from Qualtrics as text — `"Strongly disagree"`,
`"Disagree"`, … — and until you tell pandas the order, every chart and table puts them in an order
that means nothing. Three lines, once per column, and it is fixed for good.

Write the list in **increasing** order. That is the only thing you have to get right.

It also buys you something else: an ordered column can be **compared**. `df['satisfaction'] > 'neutral'`
now means something — 402 customers — where before it was comparing text alphabetically.

## 2.1 From words to numbers

An ordered column still holds words, so you cannot take its average. When you decide to treat a
Likert item as **interval** — a judgement you state in your report — turn it into the numbers 1 to 5:

In [ ]:
df['satisfaction_score'] = df['satisfaction'].cat.codes + 1

df.head()

`.cat.codes` gives each level its **position in your order**, counting from 0 — so *very unsatisfied*
is 0 and *very satisfied* is 4. The `+ 1` moves that to 1 … 5, the scale your respondents actually saw.
The new column sits at the right end of the table, next to the words it came from.

⚠️ **A missing answer becomes `-1`, so after the `+ 1` it shows up as `0`** — a number nobody chose.
`estore.csv` has no missing answers, but your survey will. Check `value_counts(sort=False)` for a 0
before you trust the average.

Now the average means something:

In [ ]:
df['satisfaction_score'].mean()

**3.09 out of 5** — just above *neutral*. That one number only exists because you **decided** the gaps
between the levels are equal. Say so when you report it.

### A second way: `pd.factorize`

You will also see this done with `pd.factorize`. It gives the same numbers — worth knowing both,
because other notebooks and AI answers use either one.

`pd.factorize` gives back **two things at once**: the codes, and the list of levels they stand for.
To keep both, put **two names on the left, separated by a comma**:

In [ ]:
codes, levels = pd.factorize(df['satisfaction'], sort=True)

print(levels)

`sort=True` puts the levels in **your** order from Part 2, so the codes again run 0 to 4.
Add 1, save it as a column, and compare it with `satisfaction_score`:

In [ ]:
df['satisfaction_factorize'] = codes + 1

df.head()

The last two columns are identical, row for row. **Same result, two ways.** `.cat.codes` is shorter,
so this course uses it from here on. A missing answer comes out as `0` here too — check before you average.

---
# Part 3 · The centre and the spread

Two numbers describe most of what a numeric column is doing: where it sits, and how spread out it is.

## 3.1 Three kinds of centre

In [ ]:
print(df['revenue'].mean())      # the average
print(df['revenue'].median())    # the middle value
print(df['revenue'].mode())      # the most common value

`112.09` · `93.05` · `0.0`.

Three answers to "where is the centre?", and they disagree — which is informative rather than annoying.

| | What it is | Its weakness |
|---|---|---|
| **Mean** | Add everything up, divide by how many | **One extreme value drags it.** The 684.17 visit pulls it up |
| **Median** | Sort them, take the middle one | Ignores how far away the extremes are |
| **Mode** | The value that occurs most often | For a continuous column it is usually meaningless — use it for nominal and ordinal |

**For a continuous column like this, the mode is usually meaningless** — money to two decimal places
rarely repeats. Here it is 0 only because **108 visits bought nothing**: a special value, not a typical
spend. That is a useful fact about the store, but it is not a centre. **Use the mode for nominal and
ordinal columns** — for example, the most common satisfaction level.

**Mean above median means a tail on the right.** Hold that thought until Part 6.

## 3.2 Spread

In [ ]:
print(df['revenue'].std())       # standard deviation
print(df['revenue'].max() - df['revenue'].min())   # the range

The **standard deviation** is roughly "how far from the mean is a typical visit" — about
**99.17** here. The **range** is 684.17, which tells you almost nothing because it is decided by
exactly two visits.

Pandas divides by **n − 1**, the rule for a **sample**. For a whole population you divide by **N** instead —
99.12 here. With 1,000 rows the difference is tiny; next week it starts to matter.

**Spread is about predictability.** If every visit spent the same, standard deviation would be
zero and you could predict any visit perfectly. The bigger the spread, the less one number tells
you — which is why reporting a mean without a spread beside it is a half-finished job.

## 3.3 Quantiles

A quantile is a cut-off: *this fraction of the data is below this value.*

In [ ]:
df['revenue'].quantile([0, 0.25, 0.5, 0.75, 1])

Those five numbers are the **five-number summary**, and the middle three are the **quartiles**:

- **Q1 = 44.92** — a quarter of visits spent less than this
- **Q2 = 93.05** — the median, half below and half above
- **Q3 = 155.94** — three quarters spent less than this

The gap between Q1 and Q3 is the **interquartile range**, the spread of the middle half of the data.
It ignores the extremes completely, which is exactly why it is useful.

In [ ]:
q1 = df['revenue'].quantile(0.25)
q3 = df['revenue'].quantile(0.75)
iqr = q3 - q1

print(iqr)
print(q3 + 1.5 * iqr)     # the upper fence

`IQR = 111.03`, and the **upper fence** is `322.49`.

That fence is a convention, not a law: anything past `Q3 + 1.5 × IQR` is conventionally flagged as an
**outlier**. **36 visits** spent more than 322.49. This is the calculation the box plot in Part 5
draws for you.

## 3.4 `describe()`, finally decoded

You have been running `describe()` since week 3 without being told what half of it meant.

In [ ]:
df['revenue'].describe()

| Row | What it is | Taught in |
|---|---|---|
| `count` | how many values are **actually there** — 1000, the same as the number of rows, so none missing | week 3 |
| `mean` | the average | §3.1 |
| `std` | the standard deviation | §3.2 |
| `min` · `max` | the extremes | week 3 |
| `25%` · `50%` · `75%` | **the quartiles** | §3.3 |

That is the whole thing. **One method, eight numbers, and you now know every one of them.**

---
# Part 4 · Better pictures of one column

## 4.1 The histogram, and what `bins` does

In [ ]:
sns.histplot(data=df, x='revenue', bins=30)
plt.show()

`bins=30` slices the range into 30 equal-width intervals and counts how many fall in each.
The tall bar on the far left is the **108 visits that bought nothing**.

Try `bins=5` and `bins=100` in the cell below. The data does not change — **your view of it does.**
Too few bins hides the shape; too many turns it into noise. There is no correct number, which is why
you should look at more than one before you believe a shape.

In [ ]:
# Your turn - try bins=5, then bins=100

## 4.2 The density plot

A histogram with the steps smoothed off. Its height says how likely values **near** that point are.

- The chance of landing in a range is the **area under the curve** over that range.
- The **total area under the curve is 100%**.

In [ ]:
sns.kdeplot(data=df, x='revenue', fill=True)
plt.show()

## 4.3 ⭐ The five things you add to every chart

A chart with no labels is not finished. These five options cover almost every formatting request you will
meet, and you should **not** memorise them — this table is your reference.

| Want | Add |
|---|---|
| A bigger figure | `plt.figure(figsize=(8, 5))` **before** the chart |
| A colour | `color='gold'` inside the chart |
| Axis labels | `plt.xlabel('Revenue per visit')` · `plt.ylabel('Number of visits')` |
| A title | `plt.title('Distribution of revenue per visit')` |
| Bars in a set order | `order=satisfaction_order` inside `countplot` |
| A **different colour per bar or box** | `hue='country', palette='deep', legend=False` |

`color=` paints everything one colour; `hue=` colours **by a column**, and `palette=` chooses the set
of colours it uses. When `hue` names the same column already on the axis, the legend just repeats the
axis labels, so turn it off with `legend=False`.

Common palettes: `'deep'` · `'Set2'` · `'viridis'` · `'pastel'`.

All together, and this is what a chart you would put in a report looks like:

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(data=df, x='revenue', bins=30, color='gold')

plt.xlabel('Revenue per visit')
plt.ylabel('Number of visits')
plt.title('Distribution of revenue per visit')
plt.show()

Note the order: **figure size first, chart in the middle, labels last, `plt.show()` to finish.**
Labels come after the chart because they are labelling something that has to exist already.

⭐ **This is an AI section.** When you want, say, a pink density plot with a title, ask for it
— *"add a title and axis labels to this seaborn density plot and make the fill pink"* — then check
that the chart still shows the same data it did before. Styling is exactly the kind of thing worth
delegating. The statistics are not.

---
# 🛠 Section 1 questions · 1 to 4

Answer in the empty cell under each question. Expected answers are shown where there is one.

**1 · Turning counts into a chance.** What share of visits came from **returning** visitors
(`revisit` = 1)? If you pick one visit at random, what is the chance it was someone's **first** visit?

```
revisit
1    0.623
0    0.377
```

_(your answer)_

In [ ]:
# Your code here

**2 · Three centres, the other way round.** Find the mean, median and mode of `pageViews`.

```
6.473
8.0
10
```

In §3.1 the mean of `revenue` was **above** its median. What is different here, which way does the
tail point, and which centre would you report?

_(your answer)_

In [ ]:
# Your code here

**3 · Read `describe()`, then use it.** Run `describe()` on `items`. Using its output, give the three
quartiles, then calculate the IQR and the upper fence.

```
IQR = 39.0, upper fence = 106.5
```

- Is the visit that bought 150 items an outlier by that rule? Would you delete it? Why, or why not?
- `min` is 0. What do the visits with 0 items mean?

_(your answer)_

In [ ]:
# Your code here

**4 · Draw a histogram of `items`, properly formatted** — a figure size, a colour, both axis labels
and a title.

In [ ]:
# Your code here

---
# ☕ Break

**Stop here.** Section 2 starts after the break.

---
---
# 📗 SECTION 2 · Groups and shape

**Parts 5 and 6, then questions 5 to 10.**

Section 1 described one column. Now you compare a column **across groups**, and ask what **shape** it has.

---
# Part 5 · One column, across groups

Almost every research question you wrote in week 4 is of this shape: **does this number differ
between these groups?** Here is how you answer it — first as a table, then as two charts.

## 5.1 The summary table

In [ ]:
df.groupby('country')['revenue'].mean()

Read it as a sentence: **group by** country, take the **revenue** column, give me the **mean** of each.

Three parts, and you can change any of them independently — `groupby('revisit')`,
`['items']`, `.median()`.

The countries come out in **alphabetical** order, because `country` is plain text. Group by
`satisfaction` instead and they come out in **your** order, because Part 2 told pandas it is ordered.

That result is a Series, not a table. For a report you want a proper two-column table with names
that make sense.

In [ ]:
summary = df.groupby('country')['revenue'].mean()

summary = summary.reset_index()
summary = summary.rename(columns={'revenue': 'Average revenue'})

summary

- **`reset_index()`** turns the group labels from row-names into a real column
- **`rename(columns={...})`** renames a column: old name on the left, new name on the right

⚠️ **Look at France: 126.10, the highest of the eight.** Before you write a word about it, go back
to `value_counts()` — France has **54 visits** in it, against 325 for the United States. A mean of 54 is
a much shakier thing than a mean of 325, and nothing in the table tells you that. Always put the group
sizes next to the group means.

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(data=df, x='country', y='revenue', errorbar=None)

plt.xticks(rotation=20)
plt.title('Mean revenue by country')
plt.show()

`barplot` computes the mean of `y` within each `x` for you. `errorbar=None` turns off the little
vertical line, which we have not covered yet.

**One bar is one number.** Two groups with wildly different spreads, or with outliers, can produce
identical bars. That is the problem the box plot solves.

## 5.3 The box plot

In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(data=df, x='country', y='revenue',
            hue='country', palette='deep', legend=False)

plt.xticks(rotation=20)
plt.title('Revenue per visit by country')
plt.show()

Every box is the five-number summary from §3.3, drawn:

| Part of the box | What it is |
|---|---|
| The line inside the box | the **median** (Q2) |
| The bottom and top of the box | **Q1** and **Q3** — the middle 50% live in the box |
| The whiskers | out to the furthest point still inside `Q1 − 1.5×IQR` … `Q3 + 1.5×IQR` |
| The dots beyond | **outliers** — past the fence you calculated in §3.3 |

**Same data as the bar chart, and it tells you far more.** The eight boxes overlap almost completely —
France's "highest" bar is a small shift inside a very wide spread. A bar chart could not have shown
you that.

**Use the box plot to look. Use the bar chart only when you have already looked.**

## 5.4 Two groups, two curves

A density plot per group, laid on top of each other:

In [ ]:
plt.figure(figsize=(9, 5))

sns.kdeplot(data=df, x='revenue', hue='revisit', fill=True, common_norm=False)

plt.title('Revenue per visit: new (0) and returning (1) visitors')
plt.show()

`hue='revisit'` draws one curve per group. **`common_norm=False`** makes each curve's area 100% on its own —
without it the two areas are shared, and the smaller group's curve is squashed flat.

Returning visitors (1) spend more, and their spending is far more spread out. The box plot shows the same
thing as boxes; this shows it as shapes.

---
# Part 6 · Shape

Centre and spread do not pin a column down. Two columns can share a mean and a standard deviation
and look nothing alike. The missing piece is **shape** — and the shape everybody compares against
is the **normal** distribution.

## 6.1 Skewness and kurtosis

In [ ]:
result = stats.describe(df['revenue'])

print("skewness:", result.skewness)
print("kurtosis:", result.kurtosis)

**Skewness ≈ 1.84. Kurtosis ≈ 5.40.** Both far from zero, which means this column is **not** close to
symmetric.

| Skewness | Shape | Where the centres sit |
|---|---|---|
| `= 0` | symmetric | mean = median = mode |
| `> 0` | **right-skewed** — a long tail to the right | mean > median |
| `< 0` | **left-skewed** — a long tail to the left | mean < median |

Check it against §3.1: mean 112.09 is well above median 93.05. Right-skewed, and the skewness number
agrees. **Two methods, same answer** — that is how you know you read it right.

**Kurtosis** measures how heavy the tails are compared with a normal distribution. Positive means
more extreme values than normal would give you; negative means fewer. 5.40 is a lot — those are the
36 big spenders past the fence.

In [ ]:
plt.figure(figsize=(6, 5))

stats.probplot(df['revenue'], plot=plt)
plt.title('QQ plot — revenue')
plt.show()

**How to read it: points on the line means normal.** That is the whole rule.

- Points hugging the straight line → close to normal
- Points rising **above** the line at the upper right → a **right tail**, more high values than normal predicts
- Points falling **below** the line at the lower left → a **left tail**, more low values than normal predicts
- Points curving away at both ends → heavy tails

This one misses the line at both ends. On the left, a flat run of points at 0 — the 108 visits that
bought nothing. On the right, the points climb far above the line — the big spenders. So:
**not normal, with a long right tail.**

## 6.3 Money is usually like this

Spending, income, prices, waiting times — anything that cannot go below zero but can go very high
tends to have a right tail. Here is a second one, much bigger.

`loans_income.csv` holds the annual incomes of **50,000 real US loan applicants**.

In [ ]:
loans = pd.read_csv('loans_income.csv')
loans = loans.rename(columns={'x': 'income'})

print(loans.shape)
print(loans['income'].mean(), loans['income'].median())

The file's single column is called `x`, which is no use to anybody, so we renamed it with the same
`rename()` from §5.1.

**Mean 68,761 against median 62,000** — the mean sits well above the median, so before drawing
anything you can already predict a right tail.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=loans, x='income', bins=40, color='seagreen')
plt.title('50,000 loan applicant incomes')
plt.show()

result = stats.describe(loans['income'])
print("skewness:", result.skewness)
print("kurtosis:", result.kurtosis)

In [ ]:
plt.figure(figsize=(6, 5))
stats.probplot(loans['income'], plot=plt)
plt.title('QQ plot — loan applicant income')
plt.show()

**Skewness 1.05** — right-skewed again. The histogram has a long tail to the right, and the QQ plot
bends away from the line at the top.

Two different files, two different kinds of money, the same shape. From now on you will be asked to
make the call *"close enough to normal"* or *"definitely not"* regularly — and for money, expect the
second.

📌 **Keep this file.** Next week it becomes the population you sample from.

## 6.4 The three checks

You now have three ways to ask whether a column is normal, and you should use them together.

| Check | Normal looks like |
|---|---|
| Compare mean and median | they are about equal |
| Skewness and kurtosis | both near 0 |
| QQ plot | points on the straight line |

**Normal means all three pass. If even one fails, it is not normal.** A column can have skewness near 0 and
still have far too many extreme values — the kurtosis and the QQ plot will show it. Always look at all three.

---
# 🛠 Section 2 questions · 5 to 10

Answer in the empty cell under each question. Expected answers are shown where there is one.

**5 · What are the skewness and kurtosis of `items`?** Is it skewed, and in which direction?

```
skewness: 0.706
kurtosis: 0.040
```

_(your answer)_

In [ ]:
# Your code here

**6 · Make a summary table of mean `revenue` by `revisit`**, with the column renamed to
something a reader would understand.

```
0     72.985172
1    135.745843
```

In [ ]:
# Your code here

**7 · Draw a box plot of `revenue` split by `satisfaction`.**

Then answer: which satisfaction level has the **highest median** revenue, and does that surprise you?

_(your answer)_

In [ ]:
# Your code here

**8 · Compare satisfaction for new and returning visitors.** Use the `satisfaction_score` column from
§2.1 to make a summary table of the **mean** score by `revisit`, with the column renamed.

```
0    3.068966
1    3.096308
```

Then answer in one sentence: is a mean a fair way to summarise a 1-to-5 answer? What did you have to
assume?

_(your answer)_

In [ ]:
# Your code here

**9 · Is `items` normally distributed?** Use all three checks from §6.4: compare the mean and median,
use your skewness and kurtosis from question 5, and draw a **QQ plot**. Then give your answer, and say
what each check told you.

_(your answer)_

In [ ]:
# Your code here

**10 · No code.** In §5.1 France had the highest mean revenue of the eight countries.

Give **two** reasons you would not put that in a report as a finding.

1. _(your answer)_
2. _(your answer)_

---
# ⭐ Finished early?

**Optional and unassessed.**

In §6.3 you saw that `loans_income` is strongly right-skewed. A standard move with skewed money data
is to take its **logarithm** and look again.

Ask your AI:

> *"Using numpy, how do I add a log-transformed column to a pandas dataframe?"*

Apply it to `loans['income']`, then **check it the way this course checks things**:

- Draw a histogram of the logged column. Is it more symmetric than before?
- Run `stats.describe` on it. Did the skewness move toward 0?
- Draw a QQ plot. Are the points closer to the line?

Then answer in one sentence: **what did you give up in exchange?**

_(your answer)_

In [ ]:
# Paste it here, run it, then check it

---
# ✅ Done — hand it in

**No cell here is designed to fail.** Run All should go straight through.

1. **Run All**, no red.
2. Save, **commit and sync** to GitHub.
3. On Moodle: **this notebook file**.
4. **Stop your Codespace**.

## What you can now do

- [ ] turn counts into probabilities with `value_counts(normalize=True)`
- [ ] give an ordinal column its proper order with `pd.Categorical` — **the Likert fix**
- [ ] turn an ordered column into the numbers 1 to 5 with `.cat.codes + 1`, and say why that is a choice
- [ ] report a centre **and** a spread, and say which centre you chose and why
- [ ] read all eight numbers in `describe()`
- [ ] calculate quartiles, the IQR and the outlier fence
- [ ] format a chart properly — size, colour, labels, title
- [ ] build a group summary table with `groupby` + `reset_index` + `rename`
- [ ] read a box plot, and say why it beats a bar chart
- [ ] judge normality three ways: mean vs median, skewness, QQ plot

## Coming up

**Next week:** why a sample can tell you about a population it came from — the bootstrap, and your
first statistical test. Keep `loans_income.csv`: it becomes the population you sample from.